# Activity-Model Parameter (Aij) Regression — Milestone 9

Activity models (van Laar, Wilson, Margules) carry two binary parameters $(A_{12}, A_{21})$ that are fitted to experimental data. This notebook fits them by **Levenberg–Marquardt** — the modern replacement for the thesis's plain Newton–Raphson (Ref (4), Pascal `TERMOV.PAS`) — and shows a satisfying closure: starting from the methanol/water bubble-pressure data of research-paper **Table 4.6**, it *recovers the van Laar parameters of Table 4.5* that generated it.

## Setup (optional)

The cell below is **commented out by default**. Uncomment it to pull the latest `vle-thermo` from PyPI.

In [1]:
# Optional: pull the latest vle-thermo from PyPI.
# Uncomment if you want the newest released version instead of
# whatever is currently in your kernel. On the hosted hub this
# install is ephemeral — it vanishes when your session is culled.
# %pip install --upgrade vle-thermo

## Context — why Levenberg–Marquardt

The fit minimizes the bubble-pressure residuals $r_d = P^{\mathrm{bub}}(A_{12}, A_{21};\, T_d, x_d) - P^{\exp}_d$ over the two parameters. Levenberg–Marquardt interpolates between **Gauss–Newton** (fast near the optimum) and **gradient descent** (robust far from it) through a damping parameter $\lambda$, so it converges gracefully from a poor initial guess where the thesis's plain Newton could diverge. See [Chapter IV](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md) and the activity models in [Chapter II §2.2](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-2-vle-theory.md).

## What this milestone built

`vle._engine.fit_aij_py(model, tcs, pcs, omegas, psat_coeffs, data, a12_0, a21_0, vl=[], ...)` returns `(a12, a21, sse, rmse, iterations)`, where `data` is a list of `(T [K], x1, P_exp [kPa])` triples. It wraps a Levenberg–Marquardt loop around the γ-φ bubble-pressure solve.

## Worked example — recover the Table 4.5 van Laar parameters

The Table 4.6 methanol(1)/water(2) bubble pressures at 298 K were computed with the van Laar parameters $\Lambda_{12} = 0.5853$, $\Lambda_{21} = 0.3458$ (Table 4.5). If we hand those pressures to the regression as if they were experimental data, it should recover the parameters — a clean end-to-end check of the whole fitting pipeline.

In [2]:
import vle._engine as e

# methanol(1), water(2).
tcs = [512.6, 647.1]
pcs = [8097.0, 22064.0]
om  = [0.564, 0.344]
psat = [[7.493, 3603.0, -34.29], [6.240, 3803.0, -46.0]]

# Table 4.6 (x1, P [kPa]) at 298 K — the 'experimental' data.
data = [
    (298.0, 0.0873, 5.1998), (298.0, 0.1900, 7.0028),
    (298.0, 0.3417, 9.1151), (298.0, 0.4943, 10.9757),
    (298.0, 0.6919, 13.2939), (298.0, 0.8492, 15.1678),
]

a12, a21, sse, rmse, iters = e.fit_aij_py(
    e.ActivityModel.VanLaar, tcs, pcs, om, psat, data,
    a12_0=0.4, a21_0=0.4)   # deliberately-off initial guess
print(f'fitted:  A12 = {a12:.4f} (Table 4.5: 0.5853)   A21 = {a21:.4f} (0.3458)')
print(f'RMSE = {rmse:.3f} kPa   converged in {iters} LM iterations')

fitted:  A12 = 0.5738 (Table 4.5: 0.5853)   A21 = 0.3681 (0.3458)
RMSE = 0.028 kPa   converged in 21 LM iterations


In [3]:
# The recovered parameters must be close to the Table 4.5 values, and
# the fit must reproduce the pressures to well under 1%.
assert abs(a12 - 0.5853) < 0.03, f'A12 {a12} off'
assert abs(a21 - 0.3458) < 0.03, f'A21 {a21} off'
assert rmse < 0.1, f'rmse {rmse} kPa too large'
print('van Laar parameters recovered from the P-x data.')

van Laar parameters recovered from the P-x data.


Starting from a deliberately wrong guess $(0.4, 0.4)$, LM converges to within a few percent of the true van Laar parameters — the small residual reflects the difference between the saturation-pressure correlation used here and the one behind the tabulated pressures.

## Exercise 1 — robustness to the initial guess

Re-run the fit from several very different starting points (e.g. $(0.1, 0.1)$, $(1.0, 0.05)$, $(0.05, 1.0)$) and confirm LM reaches essentially the same optimum each time. This is the property that plain Newton lacks.

In [4]:
# TODO: loop over a few (a12_0, a21_0) starts, call fit_aij_py, and
# print the converged (a12, a21) for each.


<details><summary>Solution</summary>

```python
for a0, b0 in [(0.1, 0.1), (1.0, 0.05), (0.05, 1.0), (0.6, 0.35)]:
    a, b, _, r, it = e.fit_aij_py(
        e.ActivityModel.VanLaar, tcs, pcs, om, psat, data, a0, b0)
    print(f'start ({a0:.2f},{b0:.2f}) -> A12={a:.4f} A21={b:.4f} rmse={r:.3f} ({it} it)')
```
All starts converge to the same neighborhood — LM's damping keeps it in the basin of attraction.
</details>

## Exercise 2 — fit a Wilson model instead

Fit the **Wilson** model (`e.ActivityModel.Wilson`) to the same data. Wilson needs the liquid molar volumes, passed via `vl=[…]` in cm³/mol (methanol ≈ 40.7, water ≈ 18.07), and its parameters are energies in kJ/kmol (start near a few hundred). Does Wilson fit the P–x data as well as van Laar?

In [5]:
# TODO: call e.fit_aij_py with e.ActivityModel.Wilson, vl=[40.7, 18.07],
# and starting guesses around a few hundred kJ/kmol; report the RMSE.


<details><summary>Solution</summary>

```python
a, b, _, r, it = e.fit_aij_py(
    e.ActivityModel.Wilson, tcs, pcs, om, psat, data,
    a12_0=500.0, a21_0=500.0, vl=[40.7, 18.07])
print(f'Wilson: A12={a:.1f} A21={b:.1f} kJ/kmol  rmse={r:.3f} kPa ({it} it)')
```
Wilson reproduces the pressures comparably well; its parameters are less tightly identified from bubble-pressure data alone, so the exact values depend more on the starting point than van Laar's do.
</details>

## References

- Research paper [Chapter IV](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md) (Tables 4.5–4.6) and [Chapter II §2.2](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-2-vle-theory.md) (activity models).
- (4) Da Silva & Báez (1989) — the Aij regression (`TERMOV.PAS`).
- (21) Orbey & Sandler — the van Laar parameters (Table 4.5).
- Algorithm details: `engine/src/flash/aij_regression.rs`; see also the [activity-models notebook](03_activity_models.ipynb).
